# 1D CNN Classifier

7-class classifier using a small (non-dilated) Conv1D stack — a plain CNN baseline to compare against the TCN's dilated version. Input: 30 x 14 window. Output: 7-class softmax.

In [ ]:
# ==========================================================
# CELL 0 — install dependencies
# ==========================================================
%pip install pandas numpy scikit-learn tensorflow joblib openpyxl matplotlib tf2onnx onnx

In [ ]:
# ==========================================================
# CELL 1 — imports
# ==========================================================
import os
import glob
import random
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import (
    Conv1D,
    Dense,
    Dropout,
    Flatten,
    GlobalAveragePooling1D,
    GRU,
    Input,
    LSTM,
)
from tensorflow.keras.models import Model


In [ ]:
# ==========================================================
# CELL 2 — model selection and configuration
# ==========================================================
MODEL_TYPE = "cnn1d"   # dense_mlp, lstm, gru, cnn1d

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

BASE_PATH = r"../../data/Motor_data_model"
SCALE_DIVISOR = 1_000_000.0

FEATURE_COLUMNS = [
    "T_slope_10s",
    "PWM",
    "Irms",
    "Vrms",
    "Current_Variance",
    "Current_Crest_Factor",
    "Vib_RMS",
    "Vib_Min",
    "Vib_Q1",
    "Vib_Median",
    "Vib_Q3",
    "Vib_Max",
    "Vib_Variance",
    "Vib_Skewness",
]

FINAL_LABELS = [
    "motor_off",
    "turning_off",
    "turning_on",
    "normal",
    "electrical_fault",
    "mechanical_fault",
    "thermal_fault",
]

LABEL_TO_ID = {
    label: index
    for index, label in enumerate(FINAL_LABELS)
}

N_FEATURES = len(FEATURE_COLUMNS)
N_CLASSES = len(FINAL_LABELS)
WINDOW_LEN = 30
WINDOW_STEP = 10
INPUT_SIZE = WINDOW_LEN * N_FEATURES
ROW_PERIOD_SEC = 0.015

N_SPLITS = 5
EPOCHS = 100
BATCH_SIZE = 64
CLIP_SCALED_INPUT = False

OUTPUT_DIR = f"supervised_{MODEL_TYPE}_output"
CV_DIR = os.path.join(OUTPUT_DIR, "cross_validation")
FINAL_DIR = os.path.join(OUTPUT_DIR, "final_model")
MANUAL_TEST_DIR = os.path.join(OUTPUT_DIR, "manual_test_results")

for folder in [CV_DIR, FINAL_DIR, MANUAL_TEST_DIR]:
    os.makedirs(folder, exist_ok=True)

print("Model:", MODEL_TYPE)
print("Features:", N_FEATURES)
print("Classes:", FINAL_LABELS)


In [ ]:
# ==========================================================
# CELL 3 — map each file path to one supervised class
# ==========================================================
def normalized_path(path):
    return str(path).replace("\\", "/").lower()


def label_for(path):
    path_text = normalized_path(path)
    stem = Path(path).stem.lower()

    if "motor_being_on" in path_text:
        return "turning_on"

    if "motor_being_off" in path_text:
        return "turning_off"

    if "transition_on" in path_text or "on_spikes" in path_text:
        return "turning_on"
    
    if "transition_off" in path_text or "off_spikes" in path_text:
        return "turning_off"

    if "/motor_off/" in path_text or "motor_off" in stem:
        return "motor_off"

    if (
        "electrical fault" in path_text
        or "electrical_fault" in path_text
        or "abnormal_current" in path_text
    ):
        return "electrical_fault"

    if (
        "mechanical fault" in path_text
        or "mechanical_fault" in path_text
        or "abnormal_vibration" in path_text
    ):
        return "mechanical_fault"

    if (
        "thermal fault" in path_text
        or "thermal_fault" in path_text
        or "abnormal_temp" in path_text
    ):
        return "thermal_fault"

    if "/normal/" in path_text or stem.startswith("normal_") or stem == "normal":
        return "normal"

    return None

In [ ]:
# ==========================================================
# CELL 4 — discover and read CSV / Excel files
# ==========================================================
def discover_files(base_path):
    files = glob.glob(
        os.path.join(base_path, "**", "*.xlsx"),
        recursive=True,
    )

    files += glob.glob(
        os.path.join(base_path, "**", "*.csv"),
        recursive=True,
    )

    return sorted(set(files))


def read_one(path):
    path = Path(path)

    if path.suffix.lower() == ".xlsx":
        df = pd.read_excel(path)
    else:
        df = pd.read_csv(path)

    missing_columns = [
        column
        for column in FEATURE_COLUMNS
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing columns in {path.name}: {missing_columns}"
        )

    clean = df[FEATURE_COLUMNS].apply(
        pd.to_numeric,
        errors="coerce",
    )

    clean = clean / SCALE_DIVISOR
    clean = clean.replace([np.inf, -np.inf], np.nan)
    clean = clean.dropna().reset_index(drop=True)

    if len(clean) < WINDOW_LEN:
        raise ValueError(
            f"{path.name} has only {len(clean)} valid rows."
        )

    return clean


In [ ]:
# ==========================================================
# CELL 5 — load labeled sessions
# ==========================================================
files = discover_files(BASE_PATH)

if not files:
    raise RuntimeError(
        f"No CSV or Excel files found under: {BASE_PATH}"
    )

sessions = []

for file_path in files:
    label = label_for(file_path)

    if label is None:
        print(
            "SKIP unrecognized:",
            os.path.relpath(file_path, BASE_PATH),
        )
        continue

    try:
        df = read_one(file_path)
    except Exception as error:
        print(
            "SKIP read error:",
            os.path.relpath(file_path, BASE_PATH),
            "|",
            error,
        )
        continue

    session = {
        "X": df.to_numpy(dtype=np.float32),
        "file": os.path.relpath(file_path, BASE_PATH),
        "label": label,
        "class_id": LABEL_TO_ID[label],
        "rows": len(df),
    }

    sessions.append(session)

if not sessions:
    raise RuntimeError("No labeled sessions were loaded.")

class_counts = pd.Series([
    session["label"]
    for session in sessions
]).value_counts().reindex(
    FINAL_LABELS,
    fill_value=0,
)

print("\nFILES PER CLASS")
print(class_counts)

missing_classes = [
    label
    for label in FINAL_LABELS
    if class_counts[label] == 0
]

if missing_classes:
    raise RuntimeError(
        f"No files found for classes: {missing_classes}"
    )



In [ ]:
# ==========================================================
# CELL 6 — scale sessions and create 30 × 14 windows
# ==========================================================
def scale_array(X_raw, scaler):
    X_scaled = scaler.transform(X_raw).astype(np.float32)

    if CLIP_SCALED_INPUT:
        X_scaled = np.clip(X_scaled, 0.0, 1.0)

    return X_scaled


def make_windows_from_array(X_raw, scaler):
    X_scaled = scale_array(X_raw, scaler)

    starts = np.arange(
        0,
        len(X_scaled) - WINDOW_LEN + 1,
        WINDOW_STEP,
        dtype=np.int64,
    )

    if len(starts) == 0:
        return None, None

    windows = np.stack([
        X_scaled[start:start + WINDOW_LEN]
        for start in starts
    ]).astype(np.float32)

    return windows, starts


def build_dataset_from_sessions(selected_sessions, scaler):
    all_windows = []
    all_labels = []
    all_files = []
    all_starts = []

    for session in selected_sessions:
        windows, starts = make_windows_from_array(
            session["X"],
            scaler,
        )

        if windows is None:
            continue

        all_windows.append(windows)
        all_labels.extend([session["class_id"]] * len(windows))
        all_files.extend([session["file"]] * len(windows))
        all_starts.extend(starts.tolist())

    if not all_windows:
        raise RuntimeError("No windows were created.")

    return (
        np.concatenate(all_windows, axis=0),
        np.asarray(all_labels, dtype=np.int64),
        np.asarray(all_files),
        np.asarray(all_starts, dtype=np.int64),
    )


In [ ]:
# ==========================================================
# CELL 7 — create class weights from training windows
# ==========================================================
def calculate_class_weights(y_train):
    present_classes = np.unique(y_train)

    weights = compute_class_weight(
        class_weight="balanced",
        classes=present_classes,
        y=y_train,
    )

    return {
        int(class_id): float(weight)
        for class_id, weight
        in zip(present_classes, weights)
    }


In [ ]:
# ==========================================================
# CELL 8 — build 1d_cnn classifier
# ==========================================================
def build_classifier():
    inputs = Input(
        shape=(WINDOW_LEN, N_FEATURES),
        name="float_input",
    )

    x = Conv1D(
        filters=32,
        kernel_size=3,
        padding="same",
        activation="relu",
        name="conv_1",
    )(inputs)

    x = Conv1D(
        filters=16,
        kernel_size=3,
        padding="same",
        activation="relu",
        name="conv_2",
    )(x)

    x = GlobalAveragePooling1D(
        name="global_average_pool",
    )(x)

    x = Dense(
        units=12,
        activation="relu",
        name="dense_1",
    )(x)

    outputs = Dense(
        units=N_CLASSES,
        activation="softmax",
        name="class_output",
    )(x)

    model = Model(
        inputs=inputs,
        outputs=outputs,
        name="cnn1d_classifier",
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=1e-3,
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    return model

In [ ]:
# ==========================================================
# CELL 9 — callbacks
# ==========================================================
def make_callbacks():
    return [
        EarlyStopping(
            monitor="val_loss",
            mode="min",
            patience=15,
            restore_best_weights=True,
            verbose=1,
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            mode="min",
            factor=0.5,
            patience=5,
            min_lr=1e-6,
            verbose=1,
        ),
    ]


temporary_model = build_classifier()
temporary_model.summary()
del temporary_model


In [ ]:
# ==========================================================
# CELL 10 — five-fold file-wise cross-validation
# ==========================================================
cv_table = pd.DataFrame({
    "label": [session["class_id"] for session in sessions],
    "group": [session["file"] for session in sessions],
})

splitter = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED,
)

cv_metrics = []
all_true = []
all_pred = []
all_window_predictions = []

for fold, (train_index, test_index) in enumerate(
    splitter.split(
        cv_table,
        cv_table["label"],
        groups=cv_table["group"],
    ),
    start=1,
):
    print("\n" + "=" * 60)
    print("FOLD", fold)
    print("=" * 60)

    train_sessions = [sessions[index] for index in train_index]
    test_sessions = [sessions[index] for index in test_index]

    rng = np.random.default_rng(SEED + fold)
    fitting_sessions = []
    validation_sessions = []

    for class_id in range(N_CLASSES):
        class_sessions = [
            session
            for session in train_sessions
            if session["class_id"] == class_id
        ]

        shuffled_ids = rng.permutation(len(class_sessions))
        number_validation = max(1, round(0.20 * len(class_sessions)))
        validation_ids = set(shuffled_ids[:number_validation])

        for local_index, session in enumerate(class_sessions):
            if local_index in validation_ids:
                validation_sessions.append(session)
            else:
                fitting_sessions.append(session)

    scaler = MinMaxScaler(feature_range=(0, 1))
    scaler.fit(
        np.concatenate([
            session["X"]
            for session in fitting_sessions
        ])
    )

    X_train, y_train, _, _ = build_dataset_from_sessions(
        fitting_sessions,
        scaler,
    )

    X_validation, y_validation, _, _ = build_dataset_from_sessions(
        validation_sessions,
        scaler,
    )

    X_test, y_test, test_files, test_starts = build_dataset_from_sessions(
        test_sessions,
        scaler,
    )

    class_weight = calculate_class_weights(y_train)

    tf.keras.backend.clear_session()
    model = build_classifier()

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_validation, y_validation),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=make_callbacks(),
        class_weight=class_weight,
        shuffle=True,
        verbose=1,
    )

    probabilities = model.predict(X_test, verbose=0)
    predictions = np.argmax(probabilities, axis=1)
    confidence = np.max(probabilities, axis=1)

    all_true.extend(y_test.tolist())
    all_pred.extend(predictions.tolist())

    fold_metrics = {
        "fold": fold,
        "accuracy": accuracy_score(y_test, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_test, predictions),
        "macro_precision": precision_score(
            y_test,
            predictions,
            average="macro",
            zero_division=0,
        ),
        "macro_recall": recall_score(
            y_test,
            predictions,
            average="macro",
            zero_division=0,
        ),
        "macro_f1": f1_score(
            y_test,
            predictions,
            average="macro",
            zero_division=0,
        ),
        "best_epoch": int(
            np.argmin(history.history["val_loss"]) + 1
        ),
    }

    cv_metrics.append(fold_metrics)

    fold_predictions = pd.DataFrame({
        "fold": fold,
        "file": test_files,
        "start_row": test_starts,
        "true_class_id": y_test,
        "true_label": [FINAL_LABELS[index] for index in y_test],
        "pred_class_id": predictions,
        "pred_label": [FINAL_LABELS[index] for index in predictions],
        "confidence": confidence,
    })

    all_window_predictions.append(fold_predictions)
    print(pd.Series(fold_metrics))



In [ ]:
# ==========================================================
# CELL 11 — combined cross-validation report
# ==========================================================
cv_metrics_df = pd.DataFrame(cv_metrics)
cv_predictions_df = pd.concat(
    all_window_predictions,
    ignore_index=True,
)

cv_metrics_df.to_csv(
    os.path.join(CV_DIR, "cv_metrics.csv"),
    index=False,
)

cv_predictions_df.to_csv(
    os.path.join(CV_DIR, "cv_window_predictions.csv"),
    index=False,
)

all_true = np.asarray(all_true, dtype=np.int64)
all_pred = np.asarray(all_pred, dtype=np.int64)

overall_metrics = {
    "accuracy": accuracy_score(all_true, all_pred),
    "balanced_accuracy": balanced_accuracy_score(all_true, all_pred),
    "macro_precision": precision_score(
        all_true,
        all_pred,
        average="macro",
        zero_division=0,
    ),
    "macro_recall": recall_score(
        all_true,
        all_pred,
        average="macro",
        zero_division=0,
    ),
    "macro_f1": f1_score(
        all_true,
        all_pred,
        average="macro",
        zero_division=0,
    ),
}

print("\nOVERALL CROSS-VALIDATION RESULTS")
print(pd.Series(overall_metrics))

print(
    classification_report(
        all_true,
        all_pred,
        labels=np.arange(N_CLASSES),
        target_names=FINAL_LABELS,
        zero_division=0,
    )
)

pd.DataFrame([overall_metrics]).to_csv(
    os.path.join(CV_DIR, "overall_metrics.csv"),
    index=False,
)

ConfusionMatrixDisplay(
    confusion_matrix(
        all_true,
        all_pred,
        labels=np.arange(N_CLASSES),
    ),
    display_labels=FINAL_LABELS,
).plot(
    values_format="d",
    xticks_rotation=45,
)

plt.title(f"{MODEL_TYPE} Cross-Validation")
plt.tight_layout()
plt.savefig(
    os.path.join(CV_DIR, "confusion_matrix.png"),
    dpi=180,
)
plt.show()


In [ ]:
# ==========================================================
# CELL 12 — final train/validation split by whole files
# ==========================================================
rng = np.random.default_rng(SEED)
final_train_sessions = []
final_validation_sessions = []

for class_id in range(N_CLASSES):
    class_sessions = [
        session
        for session in sessions
        if session["class_id"] == class_id
    ]

    shuffled_ids = rng.permutation(len(class_sessions))
    number_validation = max(1, round(0.20 * len(class_sessions)))
    validation_ids = set(shuffled_ids[:number_validation])

    for local_index, session in enumerate(class_sessions):
        if local_index in validation_ids:
            final_validation_sessions.append(session)
        else:
            final_train_sessions.append(session)

final_scaler = MinMaxScaler(feature_range=(0, 1))
final_scaler.fit(
    np.concatenate([
        session["X"]
        for session in final_train_sessions
    ])
)

X_train, y_train, _, _ = build_dataset_from_sessions(
    final_train_sessions,
    final_scaler,
)

X_validation, y_validation, validation_files, validation_starts = build_dataset_from_sessions(
    final_validation_sessions,
    final_scaler,
)

final_class_weight = calculate_class_weights(y_train)

print("Training windows:", X_train.shape)
print("Validation windows:", X_validation.shape)
print("Class weights:", final_class_weight)



In [ ]:
# ==========================================================
# CELL 13 — train final deployment model
# ==========================================================
tf.keras.backend.clear_session()
final_model = build_classifier()

final_history = final_model.fit(
    X_train,
    y_train,
    validation_data=(X_validation, y_validation),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=make_callbacks(),
    class_weight=final_class_weight,
    shuffle=True,
    verbose=1,
)

validation_probabilities = final_model.predict(
    X_validation,
    verbose=0,
)

validation_predictions = np.argmax(
    validation_probabilities,
    axis=1,
)

validation_confidence = np.max(
    validation_probabilities,
    axis=1,
)

print("\nFINAL VALIDATION REPORT")
print(
    classification_report(
        y_validation,
        validation_predictions,
        labels=np.arange(N_CLASSES),
        target_names=FINAL_LABELS,
        zero_division=0,
    )
)

print(
    "Balanced accuracy:",
    balanced_accuracy_score(
        y_validation,
        validation_predictions,
    ),
)

print(
    "Macro F1:",
    f1_score(
        y_validation,
        validation_predictions,
        average="macro",
        zero_division=0,
    ),
)

validation_results = pd.DataFrame({
    "file": validation_files,
    "start_row": validation_starts,
    "true_class_id": y_validation,
    "true_label": [FINAL_LABELS[index] for index in y_validation],
    "pred_class_id": validation_predictions,
    "pred_label": [FINAL_LABELS[index] for index in validation_predictions],
    "confidence": validation_confidence,
})

validation_results.to_csv(
    os.path.join(FINAL_DIR, "final_validation_predictions.csv"),
    index=False,
)

ConfusionMatrixDisplay(
    confusion_matrix(
        y_validation,
        validation_predictions,
        labels=np.arange(N_CLASSES),
    ),
    display_labels=FINAL_LABELS,
).plot(
    values_format="d",
    xticks_rotation=45,
)

plt.title(f"Final {MODEL_TYPE} Validation")
plt.tight_layout()
plt.savefig(
    os.path.join(FINAL_DIR, "final_validation_confusion_matrix.png"),
    dpi=180,
)
plt.show()

plt.figure(figsize=(9, 4))
plt.plot(final_history.history["loss"], label="training loss")
plt.plot(final_history.history["val_loss"], label="validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title(f"{MODEL_TYPE} Training")
plt.legend()
plt.tight_layout()
plt.savefig(
    os.path.join(FINAL_DIR, "training_loss.png"),
    dpi=150,
)
plt.show()


In [ ]:
# ==========================================================
# CELL 14 — save Keras model, scaler, labels, and metadata
# ==========================================================
KERAS_PATH = os.path.join(
    FINAL_DIR,
    f"{MODEL_TYPE}_classifier.keras",
)

SCALER_PATH = os.path.join(
    FINAL_DIR,
    f"{MODEL_TYPE}_scaler.pkl",
)

LABELS_PATH = os.path.join(
    FINAL_DIR,
    f"{MODEL_TYPE}_labels.txt",
)

METADATA_PATH = os.path.join(
    FINAL_DIR,
    f"{MODEL_TYPE}_metadata.csv",
)

final_model.save(KERAS_PATH)
joblib.dump(final_scaler, SCALER_PATH)

with open(LABELS_PATH, "w", encoding="utf-8") as file:
    file.write("\n".join(FINAL_LABELS))

pd.DataFrame({
    "feature_id": np.arange(N_FEATURES),
    "feature_name": FEATURE_COLUMNS,
    "scaler_min": final_scaler.data_min_,
    "scaler_max": final_scaler.data_max_,
}).to_csv(
    METADATA_PATH,
    index=False,
)

print("Saved:", KERAS_PATH)
print("Saved:", SCALER_PATH)
print("Saved:", LABELS_PATH)
print("Saved:", METADATA_PATH)



In [ ]:
# ==========================================================
# CELL 15 — export Float32 and dynamic-range TFLite
# ==========================================================
FLOAT32_TFLITE_PATH = os.path.join(
    FINAL_DIR,
    f"{MODEL_TYPE}_classifier_float32.tflite",
)

DYNAMIC_TFLITE_PATH = os.path.join(
    FINAL_DIR,
    f"{MODEL_TYPE}_classifier_dynamic.tflite",
)

converter = tf.lite.TFLiteConverter.from_keras_model(final_model)
float32_tflite = converter.convert()

with open(FLOAT32_TFLITE_PATH, "wb") as file:
    file.write(float32_tflite)

converter = tf.lite.TFLiteConverter.from_keras_model(final_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
dynamic_tflite = converter.convert()

with open(DYNAMIC_TFLITE_PATH, "wb") as file:
    file.write(dynamic_tflite)

print("Saved:", FLOAT32_TFLITE_PATH)
print("Saved:", DYNAMIC_TFLITE_PATH)


In [ ]:
# ==========================================================
# CELL 16 — export ONNX model
# ==========================================================
import tf2onnx

ONNX_PATH = os.path.join(
    FINAL_DIR,
    f"{MODEL_TYPE}_classifier.onnx",
)

input_signature = (
    tf.TensorSpec(
        (None, WINDOW_LEN, N_FEATURES),
        tf.float32,
        name="float_input",
    ),
)

tf2onnx.convert.from_keras(
    final_model,
    input_signature=input_signature,
    opset=13,
    output_path=ONNX_PATH,
)

print("Saved:", ONNX_PATH)



In [ ]:
# ==========================================================
# CELL 17 — export complete direct C header
# ==========================================================
HEADER_PATH = os.path.join(
    FINAL_DIR,
    f"{MODEL_TYPE}_classifier_model.h",
)


def write_float_array(file, name, values):
    values = np.asarray(values, dtype=np.float32).reshape(-1)

    file.write(
        f"static const float {name}[{len(values)}] = {{\n"
    )

    for start in range(0, len(values), 8):
        line = ", ".join(
            f"{float(value):.9g}f"
            for value in values[start:start + 8]
        )

        file.write(f"    {line},\n")

    file.write("};\n\n")


with open(HEADER_PATH, "w", encoding="utf-8") as file:
    file.write("#ifndef SUPERVISED_CLASSIFIER_MODEL_H\n")
    file.write("#define SUPERVISED_CLASSIFIER_MODEL_H\n\n")
    file.write("#include <math.h>\n\n")

    file.write(f"#define CLF_WL {WINDOW_LEN}\n")
    file.write(f"#define CLF_NF {N_FEATURES}\n")
    file.write(f"#define CLF_N_CLASSES {N_CLASSES}\n")
    file.write(f"#define CLF_DIV {SCALE_DIVISOR:.9g}f\n\n")

    write_float_array(file, "CLF_MIN", final_scaler.data_min_)
    write_float_array(file, "CLF_MAX", final_scaler.data_max_)

    if MODEL_TYPE == "dense_mlp":
        W1, B1 = final_model.get_layer("dense_1").get_weights()
        W2, B2 = final_model.get_layer("dense_2").get_weights()
        WO, BO = final_model.get_layer("class_output").get_weights()

        write_float_array(file, "CLF_W1", W1)
        write_float_array(file, "CLF_B1", B1)
        write_float_array(file, "CLF_W2", W2)
        write_float_array(file, "CLF_B2", B2)
        write_float_array(file, "CLF_WO", WO)
        write_float_array(file, "CLF_BO", BO)

        file.write("#define CLF_H1 64\n")
        file.write("#define CLF_H2 32\n\n")

    elif MODEL_TYPE == "lstm":
        K, R, B = final_model.get_layer("lstm").get_weights()
        WD, BD = final_model.get_layer("dense_1").get_weights()
        WO, BO = final_model.get_layer("class_output").get_weights()

        write_float_array(file, "CLF_K", K)
        write_float_array(file, "CLF_R", R)
        write_float_array(file, "CLF_B", B)
        write_float_array(file, "CLF_WD", WD)
        write_float_array(file, "CLF_BD", BD)
        write_float_array(file, "CLF_WO", WO)
        write_float_array(file, "CLF_BO", BO)

        file.write("#define CLF_UNITS 16\n")
        file.write("#define CLF_DENSE 16\n\n")

    elif MODEL_TYPE == "gru":
        K, R, B = final_model.get_layer("gru").get_weights()
        WD, BD = final_model.get_layer("dense_1").get_weights()
        WO, BO = final_model.get_layer("class_output").get_weights()

        write_float_array(file, "CLF_K", K)
        write_float_array(file, "CLF_R", R)
        write_float_array(file, "CLF_B", B)
        write_float_array(file, "CLF_WD", WD)
        write_float_array(file, "CLF_BD", BD)
        write_float_array(file, "CLF_WO", WO)
        write_float_array(file, "CLF_BO", BO)

        file.write("#define CLF_UNITS 16\n")
        file.write("#define CLF_DENSE 16\n\n")

    elif MODEL_TYPE == "cnn1d":
        K1, B1 = final_model.get_layer("conv_1").get_weights()
        K2, B2 = final_model.get_layer("conv_2").get_weights()
        WD, BD = final_model.get_layer("dense_1").get_weights()
        WO, BO = final_model.get_layer("class_output").get_weights()

        write_float_array(file, "CLF_K1", K1)
        write_float_array(file, "CLF_B1", B1)
        write_float_array(file, "CLF_K2", K2)
        write_float_array(file, "CLF_B2", B2)
        write_float_array(file, "CLF_WD", WD)
        write_float_array(file, "CLF_BD", BD)
        write_float_array(file, "CLF_WO", WO)
        write_float_array(file, "CLF_BO", BO)

        file.write("#define CLF_F1 16\n")
        file.write("#define CLF_F2 8\n")
        file.write("#define CLF_DENSE 16\n\n")

    file.write(r'''
static inline float clf_sigmoid(float x)
{
    return 1.0f / (1.0f + expf(-x));
}


static inline float clf_relu(float x)
{
    return x > 0.0f ? x : 0.0f;
}


static inline void clf_softmax(float x[CLF_N_CLASSES])
{
    float maximum = x[0];

    for (int i = 1; i < CLF_N_CLASSES; i++)
    {
        if (x[i] > maximum)
        {
            maximum = x[i];
        }
    }

    float sum = 0.0f;

    for (int i = 0; i < CLF_N_CLASSES; i++)
    {
        x[i] = expf(x[i] - maximum);
        sum += x[i];
    }

    for (int i = 0; i < CLF_N_CLASSES; i++)
    {
        x[i] /= sum;
    }
}
''')

    if MODEL_TYPE == "dense_mlp":
        file.write(r'''
static inline int classifier_predict(
    const float raw[CLF_WL][CLF_NF],
    float probability[CLF_N_CLASSES],
    float *confidence)
{
    float input[CLF_WL * CLF_NF];
    float h1[CLF_H1];
    float h2[CLF_H2];

    for (int t = 0; t < CLF_WL; t++)
    {
        for (int f = 0; f < CLF_NF; f++)
        {
            float physical = raw[t][f] / CLF_DIV;
            float range = CLF_MAX[f] - CLF_MIN[f];

            input[t * CLF_NF + f] =
                range > 0.0f
                ? (physical - CLF_MIN[f]) / range
                : 0.0f;
        }
    }

    for (int j = 0; j < CLF_H1; j++)
    {
        float sum = CLF_B1[j];

        for (int i = 0; i < CLF_WL * CLF_NF; i++)
        {
            sum += input[i] * CLF_W1[i * CLF_H1 + j];
        }

        h1[j] = clf_relu(sum);
    }

    for (int j = 0; j < CLF_H2; j++)
    {
        float sum = CLF_B2[j];

        for (int i = 0; i < CLF_H1; i++)
        {
            sum += h1[i] * CLF_W2[i * CLF_H2 + j];
        }

        h2[j] = clf_relu(sum);
    }

    for (int j = 0; j < CLF_N_CLASSES; j++)
    {
        float sum = CLF_BO[j];

        for (int i = 0; i < CLF_H2; i++)
        {
            sum += h2[i] * CLF_WO[i * CLF_N_CLASSES + j];
        }

        probability[j] = sum;
    }

    clf_softmax(probability);

    int class_id = 0;
    *confidence = probability[0];

    for (int j = 1; j < CLF_N_CLASSES; j++)
    {
        if (probability[j] > *confidence)
        {
            *confidence = probability[j];
            class_id = j;
        }
    }

    return class_id;
}
''')

    elif MODEL_TYPE == "lstm":
        file.write(r'''
static inline int classifier_predict(
    const float raw[CLF_WL][CLF_NF],
    float probability[CLF_N_CLASSES],
    float *confidence)
{
    float h[CLF_UNITS] = {0};
    float c[CLF_UNITS] = {0};
    float gates[4 * CLF_UNITS];
    float dense[CLF_DENSE];

    for (int t = 0; t < CLF_WL; t++)
    {
        float x[CLF_NF];

        for (int f = 0; f < CLF_NF; f++)
        {
            float physical = raw[t][f] / CLF_DIV;
            float range = CLF_MAX[f] - CLF_MIN[f];

            x[f] = range > 0.0f
                ? (physical - CLF_MIN[f]) / range
                : 0.0f;
        }

        for (int gate = 0; gate < 4 * CLF_UNITS; gate++)
        {
            float sum = CLF_B[gate];

            for (int f = 0; f < CLF_NF; f++)
            {
                sum += x[f] * CLF_K[f * 4 * CLF_UNITS + gate];
            }

            for (int unit = 0; unit < CLF_UNITS; unit++)
            {
                sum += h[unit] * CLF_R[unit * 4 * CLF_UNITS + gate];
            }

            gates[gate] = sum;
        }

        for (int unit = 0; unit < CLF_UNITS; unit++)
        {
            float input_gate = clf_sigmoid(gates[unit]);
            float forget_gate = clf_sigmoid(gates[CLF_UNITS + unit]);
            float candidate = tanhf(gates[2 * CLF_UNITS + unit]);
            float output_gate = clf_sigmoid(gates[3 * CLF_UNITS + unit]);

            c[unit] = forget_gate * c[unit] + input_gate * candidate;
            h[unit] = output_gate * tanhf(c[unit]);
        }
    }

    for (int j = 0; j < CLF_DENSE; j++)
    {
        float sum = CLF_BD[j];

        for (int i = 0; i < CLF_UNITS; i++)
        {
            sum += h[i] * CLF_WD[i * CLF_DENSE + j];
        }

        dense[j] = clf_relu(sum);
    }

    for (int j = 0; j < CLF_N_CLASSES; j++)
    {
        float sum = CLF_BO[j];

        for (int i = 0; i < CLF_DENSE; i++)
        {
            sum += dense[i] * CLF_WO[i * CLF_N_CLASSES + j];
        }

        probability[j] = sum;
    }

    clf_softmax(probability);

    int class_id = 0;
    *confidence = probability[0];

    for (int j = 1; j < CLF_N_CLASSES; j++)
    {
        if (probability[j] > *confidence)
        {
            *confidence = probability[j];
            class_id = j;
        }
    }

    return class_id;
}
''')

    elif MODEL_TYPE == "gru":
        file.write(r'''
static inline int classifier_predict(
    const float raw[CLF_WL][CLF_NF],
    float probability[CLF_N_CLASSES],
    float *confidence)
{
    float h[CLF_UNITS] = {0};
    float previous_h[CLF_UNITS];
    float z[CLF_UNITS];
    float r[CLF_UNITS];
    float candidate[CLF_UNITS];
    float dense[CLF_DENSE];

    for (int t = 0; t < CLF_WL; t++)
    {
        float x[CLF_NF];

        for (int f = 0; f < CLF_NF; f++)
        {
            float physical = raw[t][f] / CLF_DIV;
            float range = CLF_MAX[f] - CLF_MIN[f];

            x[f] = range > 0.0f
                ? (physical - CLF_MIN[f]) / range
                : 0.0f;
        }

        for (int unit = 0; unit < CLF_UNITS; unit++)
        {
            previous_h[unit] = h[unit];
        }

        for (int unit = 0; unit < CLF_UNITS; unit++)
        {
            float sum_z = CLF_B[unit];
            float sum_r = CLF_B[CLF_UNITS + unit];

            for (int f = 0; f < CLF_NF; f++)
            {
                sum_z += x[f] * CLF_K[f * 3 * CLF_UNITS + unit];
                sum_r += x[f] * CLF_K[f * 3 * CLF_UNITS + CLF_UNITS + unit];
            }

            for (int i = 0; i < CLF_UNITS; i++)
            {
                sum_z += previous_h[i] * CLF_R[i * 3 * CLF_UNITS + unit];
                sum_r += previous_h[i] * CLF_R[i * 3 * CLF_UNITS + CLF_UNITS + unit];
            }

            z[unit] = clf_sigmoid(sum_z);
            r[unit] = clf_sigmoid(sum_r);
        }

        for (int unit = 0; unit < CLF_UNITS; unit++)
        {
            float sum_h = CLF_B[2 * CLF_UNITS + unit];

            for (int f = 0; f < CLF_NF; f++)
            {
                sum_h += x[f] * CLF_K[f * 3 * CLF_UNITS + 2 * CLF_UNITS + unit];
            }

            for (int i = 0; i < CLF_UNITS; i++)
            {
                sum_h +=
                    (r[i] * previous_h[i])
                    * CLF_R[i * 3 * CLF_UNITS + 2 * CLF_UNITS + unit];
            }

            candidate[unit] = tanhf(sum_h);
        }

        for (int unit = 0; unit < CLF_UNITS; unit++)
        {
            h[unit] =
                z[unit] * previous_h[unit]
                + (1.0f - z[unit]) * candidate[unit];
        }
    }

    for (int j = 0; j < CLF_DENSE; j++)
    {
        float sum = CLF_BD[j];

        for (int i = 0; i < CLF_UNITS; i++)
        {
            sum += h[i] * CLF_WD[i * CLF_DENSE + j];
        }

        dense[j] = clf_relu(sum);
    }

    for (int j = 0; j < CLF_N_CLASSES; j++)
    {
        float sum = CLF_BO[j];

        for (int i = 0; i < CLF_DENSE; i++)
        {
            sum += dense[i] * CLF_WO[i * CLF_N_CLASSES + j];
        }

        probability[j] = sum;
    }

    clf_softmax(probability);

    int class_id = 0;
    *confidence = probability[0];

    for (int j = 1; j < CLF_N_CLASSES; j++)
    {
        if (probability[j] > *confidence)
        {
            *confidence = probability[j];
            class_id = j;
        }
    }

    return class_id;
}
''')

    elif MODEL_TYPE == "cnn1d":
        file.write(r'''
static inline void clf_conv_same(
    const float *input,
    int input_channels,
    const float *kernel,
    const float *bias,
    int output_channels,
    float *output)
{
    for (int t = 0; t < CLF_WL; t++)
    {
        for (int oc = 0; oc < output_channels; oc++)
        {
            float sum = bias[oc];

            for (int k = 0; k < 3; k++)
            {
                int source_t = t + k - 1;

                if (source_t < 0 || source_t >= CLF_WL)
                {
                    continue;
                }

                for (int ic = 0; ic < input_channels; ic++)
                {
                    int input_index = source_t * input_channels + ic;
                    int kernel_index =
                        (k * input_channels + ic) * output_channels + oc;

                    sum += input[input_index] * kernel[kernel_index];
                }
            }

            output[t * output_channels + oc] = clf_relu(sum);
        }
    }
}


static inline int classifier_predict(
    const float raw[CLF_WL][CLF_NF],
    float probability[CLF_N_CLASSES],
    float *confidence)
{
    float input[CLF_WL * CLF_NF];
    float conv_1[CLF_WL * CLF_F1];
    float conv_2[CLF_WL * CLF_F2];
    float pooled[CLF_F2];
    float dense[CLF_DENSE];

    for (int t = 0; t < CLF_WL; t++)
    {
        for (int f = 0; f < CLF_NF; f++)
        {
            float physical = raw[t][f] / CLF_DIV;
            float range = CLF_MAX[f] - CLF_MIN[f];

            input[t * CLF_NF + f] =
                range > 0.0f
                ? (physical - CLF_MIN[f]) / range
                : 0.0f;
        }
    }

    clf_conv_same(
        input,
        CLF_NF,
        CLF_K1,
        CLF_B1,
        CLF_F1,
        conv_1
    );

    clf_conv_same(
        conv_1,
        CLF_F1,
        CLF_K2,
        CLF_B2,
        CLF_F2,
        conv_2
    );

    for (int channel = 0; channel < CLF_F2; channel++)
    {
        float sum = 0.0f;

        for (int t = 0; t < CLF_WL; t++)
        {
            sum += conv_2[t * CLF_F2 + channel];
        }

        pooled[channel] = sum / CLF_WL;
    }

    for (int j = 0; j < CLF_DENSE; j++)
    {
        float sum = CLF_BD[j];

        for (int i = 0; i < CLF_F2; i++)
        {
            sum += pooled[i] * CLF_WD[i * CLF_DENSE + j];
        }

        dense[j] = clf_relu(sum);
    }

    for (int j = 0; j < CLF_N_CLASSES; j++)
    {
        float sum = CLF_BO[j];

        for (int i = 0; i < CLF_DENSE; i++)
        {
            sum += dense[i] * CLF_WO[i * CLF_N_CLASSES + j];
        }

        probability[j] = sum;
    }

    clf_softmax(probability);

    int class_id = 0;
    *confidence = probability[0];

    for (int j = 1; j < CLF_N_CLASSES; j++)
    {
        if (probability[j] > *confidence)
        {
            *confidence = probability[j];
            class_id = j;
        }
    }

    return class_id;
}
''')

    file.write("\n#endif\n")

print("Saved:", HEADER_PATH)
print(
    "Header size KB:",
    round(os.path.getsize(HEADER_PATH) / 1024, 2),
)

In [ ]:
# ==========================================================
# CELL — export and download CNN1D parameter header only
# Run after CELL 12, where final_scaler is fitted
# ==========================================================
import os
import numpy as np

PARAM_HEADER_PATH = os.path.join(
    FINAL_DIR,
    "cnn1d_classifier_params.h",
)

def c_float(value):
    text = f"{float(np.float32(value)):.9g}"
    if ("." not in text) and ("e" not in text.lower()):
        text += ".0"
    return text + "f"

def write_c_array(file, name, values):
    values = np.asarray(values, dtype=np.float32).reshape(-1)

    file.write(
        f"static const float {name}[CLF_N_FEATURES] = {{\n"
    )

    for start in range(0, len(values), 4):
        row = ", ".join(
            c_float(value)
            for value in values[start:start + 4]
        )
        file.write(f"    {row},\n")

    file.write("};\n\n")

with open(PARAM_HEADER_PATH, "w", encoding="utf-8") as file:
    file.write("#ifndef CNN1D_CLASSIFIER_PARAMS_H\n")
    file.write("#define CNN1D_CLASSIFIER_PARAMS_H\n\n")

    file.write(f"#define CLF_WINDOW_LEN  {WINDOW_LEN}U\n")
    file.write(f"#define CLF_N_FEATURES  {N_FEATURES}U\n")
    file.write(f"#define CLF_N_CLASSES   {N_CLASSES}U\n")
    file.write(f"#define CLF_WINDOW_STEP {WINDOW_STEP}U\n")
    file.write(
        f"#define CLF_SCALE_DIVISOR {c_float(SCALE_DIVISOR)}\n\n"
    )

    file.write(
        "static const char *const "
        "CLF_LABELS[CLF_N_CLASSES] = {\n"
    )
    for label in FINAL_LABELS:
        file.write(f'    "{label}",\n')
    file.write("};\n\n")

    file.write(
        "static const char *const "
        "CLF_FEATURE_NAMES[CLF_N_FEATURES] = {\n"
    )
    for feature in FEATURE_COLUMNS:
        file.write(f'    "{feature}",\n')
    file.write("};\n\n")

    write_c_array(file, "CLF_MIN", final_scaler.data_min_)
    write_c_array(file, "CLF_MAX", final_scaler.data_max_)

    file.write("#endif /* CNN1D_CLASSIFIER_PARAMS_H */\n")

print("Created:", PARAM_HEADER_PATH)

try:
    from google.colab import files
    files.download(PARAM_HEADER_PATH)
except ImportError:
    from IPython.display import FileLink, display
    display(FileLink(PARAM_HEADER_PATH))

In [ ]:
# ==========================================================
# CELL 18 — manual classification test
# ==========================================================
def manual_classification_prediction(test_file):
    model = tf.keras.models.load_model(KERAS_PATH)
    scaler = joblib.load(SCALER_PATH)

    test_df = read_one(test_file)

    X_test, starts = make_windows_from_array(
        test_df.to_numpy(dtype=np.float32),
        scaler,
    )

    probabilities = model.predict(X_test, verbose=0)
    predictions = np.argmax(probabilities, axis=1)
    confidence = np.max(probabilities, axis=1)

    result = pd.DataFrame({
        "window": np.arange(len(X_test)),
        "start_row": starts,
        "time_sec": (
            starts + WINDOW_LEN / 2.0
        ) * ROW_PERIOD_SEC,
        "pred_class_id": predictions,
        "pred_label": [FINAL_LABELS[index] for index in predictions],
        "confidence": confidence,
    })

    for class_id, label in enumerate(FINAL_LABELS):
        result[f"probability_{label}"] = probabilities[:, class_id]

    majority_class_id = int(
        result["pred_class_id"].value_counts().idxmax()
    )

    majority_label = FINAL_LABELS[majority_class_id]
    stem = Path(test_file).stem

    output_csv = os.path.join(
        MANUAL_TEST_DIR,
        f"manual_test_{stem}.csv",
    )

    result.to_csv(output_csv, index=False)

    print("\n" + "=" * 60)
    print(f"MANUAL {MODEL_TYPE} TEST")
    print("=" * 60)
    print("File:", test_file)
    print("Total windows:", len(result))
    print("Majority prediction:", majority_label)
    print("Mean confidence:", float(np.mean(confidence)))
    print("Saved CSV:", output_csv)

    print("\nWindow predictions:")
    print(result["pred_label"].value_counts())

    plt.figure(figsize=(12, 4))
    plt.step(
        result["time_sec"],
        result["pred_class_id"],
        where="mid",
    )
    plt.yticks(np.arange(N_CLASSES), FINAL_LABELS)
    plt.xlabel("Time [s]")
    plt.ylabel("Predicted class")
    plt.title(f"{MODEL_TYPE} — {stem}")
    plt.tight_layout()
    plt.savefig(
        os.path.join(
            MANUAL_TEST_DIR,
            f"manual_test_{stem}_timeline.png",
        ),
        dpi=150,
    )
    plt.show()

    return result



In [ ]:
# ==========================================================
# CELL 19 — select manual test file
# ==========================================================
if __name__ == "__main__":
    import tkinter as tk
    from tkinter import filedialog

    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)

    TEST_FILE = filedialog.askopenfilename(
        title="Select motor CSV or Excel file",
        filetypes=[
            ("Excel files", "*.xlsx"),
            ("CSV files", "*.csv"),
            ("All files", "*.*"),
        ],
    )

    root.destroy()

    if TEST_FILE:
        manual_result = manual_classification_prediction(TEST_FILE)
    else:
        print("No file selected.")